# Flight Delay Prediction — Data Cleaning
**Author:** Justin Engwall  
**Date:** June 2026  
**Course:** DATASCI 207  

## Overview
This notebook handles initial data cleaning for the BTS Airline On-Time Statistics dataset.
Tasks include:
- Loading raw data from the TranStats database
- Inspecting shape, dtypes, and missing values
- Dropping or imputing missing entries
- Standardizing column names
- Exporting a cleaned dataset for EDA and modeling



## Import libraries

In [1]:
import numpy as np
import pandas as pd
import glob
import os

/Users/justin/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/justin/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## Data ingestion
### Data Sources
Flight data is from Jan 1, 2022 to Jan 1, 2026.     
Departure and Arrival Aiport: Atlanta, GA: Hartsfield-Jackson Atlanta International (ATL).   
Airlines: Delta (DL), American (AA), Southwest (WN), United (UA)  

U.S. Department of Transportation — Bureau of Transportation Statistics (BTS)  
https://www.transtats.bts.gov/ontime

Hourly weather observations - NOAA Global Historical Climatology Network  
https://www.ncei.noaa.gov/access/search/data-search/global-historical-climatology-network-hourly

FAA Aircraft Registry - all registered US aircraft (type, manufacturer, year, engine type, # of seats)  
https://registry.faa.gov/aircraftinquiry/Search/NNumberInquiry

FAA N-Number - lookup for individual aircraft  
https://www.faa.gov/licenses_certificates/aircraft_certification/aircraft_registry/releasable_aircraft_download?utm_source=chatgpt.com

In [2]:
# set location for directory
os.chdir('/Users/justin/DATASCI207/207-Summer26-FinalProject-MLModel/datasets')

# airlines in dataset
airlines = ['AA', 'DL', 'WN', 'UA']

# load ATL arrival data
arrivals_df = []
# read each .csv arrival for each airline
for airline in airlines:
    df = pd.read_csv(f"ATL_{airline}_Arrivals.csv", skiprows=7)
    arrivals_df.append(df)

atl_arrivals = pd.concat(arrivals_df, ignore_index=True)

# load ATL departure data
departures_df = []
# read each .csv arrival for each airline
for airline in airlines:
    df = pd.read_csv(f"ATL_{airline}_Departures.csv", skiprows=7)
    departures_df.append(df)

atl_departures = pd.concat(departures_df, ignore_index=True)

In [3]:
# take a peak at the arrival data
# atl_arrivals.head
print("ATL arrivals shape:", atl_arrivals.shape)
print("ATL departures shape:", atl_departures.shape)
print("The data type of arrivals:", atl_arrivals.dtypes)

ATL arrivals shape: (1161149, 17)
ATL departures shape: (1161257, 17)
The data type of arrivals: Carrier Code                                    str
Date (MM/DD/YYYY)                               str
Flight Number                               float64
Tail Number                                     str
Origin Airport                                  str
Scheduled Arrival Time                          str
Actual Arrival Time                             str
Scheduled Elapsed Time (Minutes)            float64
Actual Elapsed Time (Minutes)               float64
Arrival Delay (Minutes)                     float64
Wheels-on Time                                  str
Taxi-In time (Minutes)                      float64
Delay Carrier (Minutes)                     float64
Delay Weather (Minutes)                     float64
Delay National Aviation System (Minutes)    float64
Delay Security (Minutes)                    float64
Delay Late Aircraft Arrival (Minutes)       float64
dtype: object


In [4]:
## arrival conversion
# replace hour 24:00 with 00:00 before converting to datetime
atl_arrivals['Actual Arrival Time'] = atl_arrivals['Actual Arrival Time'].str.replace('24:00', '00:00')
atl_arrivals['Wheels-on Time'] = atl_arrivals['Wheels-on Time'].str.replace('24:00', '00:00')
atl_arrivals['Scheduled Arrival Time'] = atl_arrivals['Scheduled Arrival Time'].str.replace('24:00', '00:00')

# convert arrival time string to datetime
atl_arrivals['Date (MM/DD/YYYY)'] = pd.to_datetime(atl_arrivals['Date (MM/DD/YYYY)'])
atl_arrivals['Actual Arrival Time'] = pd.to_datetime(atl_arrivals['Actual Arrival Time'], format='%H:%M').dt.time
atl_arrivals['Wheels-on Time'] = pd.to_datetime(atl_arrivals['Wheels-on Time'], format='%H:%M').dt.time
atl_arrivals['Scheduled Arrival Time'] = pd.to_datetime(atl_arrivals['Scheduled Arrival Time'], format='%H:%M').dt.time

## departure conversion
atl_departures['Date (MM/DD/YYYY)'] = pd.to_datetime(atl_departures['Date (MM/DD/YYYY)'])
atl_departures['Scheduled departure time'] = atl_departures['Scheduled departure time'].str.replace('24:00', '00:00')
atl_departures['Actual departure time'] = atl_departures['Actual departure time'].str.replace('24:00', '00:00')
atl_departures['Wheels-off time'] = atl_departures['Wheels-off time'].str.replace('24:00', '00:00')

# covert departure time string to datetime
atl_departures['Scheduled departure time'] = pd.to_datetime(atl_departures['Scheduled departure time'], format='%H:%M').dt.time
atl_departures['Actual departure time'] = pd.to_datetime(atl_departures['Actual departure time'], format='%H:%M').dt.time
atl_departures['Wheels-off time'] = pd.to_datetime(atl_departures['Wheels-off time'], format='%H:%M').dt.time

## verify the conversion
print(atl_arrivals[['Date (MM/DD/YYYY)', 'Scheduled Arrival Time', 'Actual Arrival Time']].dtypes)
print(atl_arrivals[['Date (MM/DD/YYYY)', 'Scheduled Arrival Time', 'Actual Arrival Time']].head())
print(atl_departures[['Date (MM/DD/YYYY)', 'Scheduled departure time', 'Actual departure time']].dtypes)
print(atl_departures[['Date (MM/DD/YYYY)', 'Scheduled departure time', 'Actual departure time']].head())
print(atl_arrivals['Date (MM/DD/YYYY)'].dtype)
print(atl_departures['Date (MM/DD/YYYY)'].dtype)

Date (MM/DD/YYYY)         datetime64[us]
Scheduled Arrival Time            object
Actual Arrival Time               object
dtype: object
  Date (MM/DD/YYYY) Scheduled Arrival Time Actual Arrival Time
0        2022-01-01               23:29:00            23:19:00
1        2022-01-01               23:40:00            23:31:00
2        2022-01-01               21:53:00            21:42:00
3        2022-01-01               15:52:00            16:57:00
4        2022-01-01               13:25:00            13:43:00
Date (MM/DD/YYYY)           datetime64[us]
Scheduled departure time            object
Actual departure time               object
dtype: object
  Date (MM/DD/YYYY) Scheduled departure time Actual departure time
0        2022-01-01                 08:36:00              08:32:00
1        2022-01-01                 14:05:00              14:24:00
2        2022-01-01                 17:00:00              17:51:00
3        2022-01-01                 16:29:00              16:24:00
4      

In [5]:
print(atl_departures.columns.tolist())


['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Tail Number', 'Destination Airport', 'Scheduled departure time', 'Actual departure time', 'Scheduled elapsed time (Minutes)', 'Actual elapsed time (Minutes)', 'Departure delay (Minutes)', 'Wheels-off time', 'Taxi-Out time (Minutes)', 'Delay Carrier (Minutes)', 'Delay Weather (Minutes)', 'Delay National Aviation System (Minutes)', 'Delay Security (Minutes)', 'Delay Late Aircraft Arrival (Minutes)']
